# Evaluation and robustness
Train in the terminal with `python -m medical_gradcam train`, then use this notebook for results.
Select the project's `.venv` kernel and run from the project directory. No training happens here.

The notebook loads the best checkpoint saved by terminal training.
Check the selected checkpoint below. Use validation to choose settings; reserve the test set
for final comparisons. Heatmaps are model attributions, not lesion annotations.


In [ ]:
import json
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
from IPython.display import Image as DisplayImage, Markdown, display

from medical_gradcam.config import Config
from medical_gradcam.evaluate import evaluate
from medical_gradcam.explain import explain
from medical_gradcam.robustness import robustness

config = Config.load("config.json")
run_dir = Path(config.run_dir)
checkpoint = run_dir / "best.pt"
if not checkpoint.exists():
    raise FileNotFoundError("No checkpoint found. Train in the terminal first.")
print("Checkpoint:", checkpoint.resolve())


## Training curves
Read the history saved by terminal training. Curves appear after training has saved its history.


In [ ]:
history_path = run_dir / "history.json"
if history_path.exists():
    history = json.loads(history_path.read_text(encoding="utf-8"))
    epochs = [row["epoch"] for row in history]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for split in ("train", "val"):
        axes[0].plot(epochs, [row[split]["loss"] for row in history], label=split)
        axes[1].plot(epochs, [row[split]["accuracy"] for row in history], label=split)
    for ax, title in zip(axes, ("Loss", "Accuracy")):
        ax.set(title=title, xlabel="Epoch")
        ax.legend()
    fig.tight_layout()
    display(fig)
    plt.close(fig)
else:
    print("No training history for this checkpoint. Curves appear after terminal training.")


## Classification metrics
This cell evaluates every test image and displays the saved confusion matrix inline.
To evaluate validation instead, set `evaluation_split = "val"`.


In [ ]:
evaluation_split = "test"
metrics = evaluate(config, checkpoint, split=evaluation_split)
print(f"Accuracy: {metrics['accuracy']:.2%}")
print("ROC-AUC:", metrics["roc_auc"])
print("Average precision:", metrics["average_precision"])
print("Brier score:", metrics["brier_score"])
print("Calibration error:", metrics["expected_calibration_error"])
print("Patient separation verified:", metrics["patient_separation_verified"])
pprint(metrics["classification_report"])
evaluation_dir = run_dir / ("evaluation" if evaluation_split == "test" else "validation")
display(DisplayImage(filename=str(evaluation_dir / "confusion_matrix.png")))


## Grad-CAM on one example
By default, choose the first incorrect prediction if there is one. Change `image_path`
to inspect another image. Compare methods by changing `method` to `"gradcam++"`.


In [ ]:
predictions = json.loads((evaluation_dir / "predictions.json").read_text(encoding="utf-8"))
incorrect = [row for row in predictions if row["true"] != row["predicted"]]
case = (incorrect or predictions)[0]
image_path = Path(config.data_dir) / case["path"]
method = "gradcam"
print("Image:", image_path)
print("True:", case["true"], "| Predicted:", case["predicted"])
explain(config, checkpoint, image_path, method=method)
comparison = run_dir / "explanations" / image_path.stem / method / "comparison.png"
display(DisplayImage(filename=str(comparison)))


## Robustness experiments
Compare border masking, equal-area central masking, brightness changes and translation.
The default is a seeded sample of 100 validation images for a new checkpoint.
Prediction changes alone do not prove shortcut learning.


In [ ]:
robustness_split = "val"
robustness_limit = 100
robustness(config, checkpoint, split=robustness_split, limit=robustness_limit)
robustness_dir = run_dir / "robustness" / robustness_split
results = json.loads((robustness_dir / "results.json").read_text(encoding="utf-8"))
print("Sample size:", results["sample_count"], "of", results["total_available"])
print(results["limitations"])
rows = ["| Variant | Accuracy | Prediction flip rate | Mean probability change |",
        "|---|---:|---:|---:|"]
for name, values in results["summary"].items():
    rows.append(f"| {name} | {values['accuracy']:.2%} | {values['prediction_flip_rate']:.2%} | "
                f"{values['mean_original_class_probability_change']:+.4f} |")
display(Markdown("\n".join(rows)))


In [ ]:
for variant in results["summary"]:
    display(Markdown(f"**{variant}**"))
    display(DisplayImage(filename=str(robustness_dir / f"example_{variant}.png"), width=260))
